# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

1. `1 1.3333333333333333`. В Series один пропуск, поэтому `isna().sum()` возвращает 1. После замены пропуска на 0 среднее равно (1 + 0 + 3) / 3 = 1.333...
2. `2`. После удаления пробелов по краям и приведения к нижнему регистру получаем `["a", "a", "b"]`, то есть два уникальных значения.
3. `3`. `drop_duplicates()` возвращает новую таблицу, но результат не сохранён в `df` и исходный DataFrame не изменён.
4. `1005`. Пробел в `"1 000"` удаляется, строки превращаются в числа 1000 и 5, сумма равна 1005.
5. `[9, 10]`. Первая дата относится к сентябрю (месяц 9), вторая — к октябрю (месяц 10).

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1. Функция очистки
import pandas as pd
import numpy as np

# Приводим варианты написания к шести единым названиям.
CITY_MAP = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "нур-султан": "Астана", "нурсултан": "Астана", "astana": "Астана", "nur-sultan": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда", "karaganda": "Караганда",
    "актобе": "Актобе", "aktobe": "Актобе",
    "тараз": "Тараз", "taraz": "Тараз"
}

def clean(path):
    data = pd.read_csv(path)
    print("Исходные строки:", len(data))

    # Суммы: убираем пробелы-разделители тысяч и переводим в числа.
    data["amount"] = pd.to_numeric(
        data["amount"].astype("string").str.replace(r"\\s+", "", regex=True),
        errors="coerce"
    )
    data = data.dropna(subset=["amount"])
    print("После очистки amount:", len(data))

    # Города: убираем лишние пробелы, нормализуем регистр и варианты названий.
    city_normalized = data["city"].astype("string").str.strip().str.lower()
    data["city"] = city_normalized.map(CITY_MAP)
    data = data.dropna(subset=["city"])
    print("После нормализации city:", len(data))

    # Удаляем повторные записи одного заказа.
    data = data.drop_duplicates(subset=["order_id"], keep="first")
    print("После удаления дублей order_id:", len(data))

    # Преобразуем дату и удаляем строки с некорректной датой.
    data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")
    data = data.dropna(subset=["order_date"])
    print("После преобразования дат:", len(data))

    # Тестовые записи исключаем, отменённые заказы сохраняем.
    data = data[data["customer_id"].astype("string").str.strip().str.upper() != "TEST"].copy()
    print("После исключения TEST:", len(data))

    # Проверки качества данных.
    assert data["amount"].isna().sum() == 0
    assert data["city"].isna().sum() == 0
    assert data["order_id"].is_unique
    return data

clean_df = clean("orders.csv")
print("\\nИтоговый размер:", clean_df.shape)
print("Уникальные города:", sorted(clean_df["city"].unique()))
print("Итого строк:", len(clean_df))

Исходные строки: 5463
После очистки amount: 5018
После нормализации city: 4883
После удаления дублей order_id: 4729
После преобразования дат: 4729
После исключения TEST: 4726
\nИтоговый размер: (4726, 8)
Уникальные города: ['Актобе', 'Алматы', 'Астана', 'Караганда', 'Шымкент']
Итого строк: 4726


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# 2. Выручка по чистым данным
# Выручка — сумма amount только для оплаченных заказов.
paid_clean = clean_df[clean_df["status"].astype("string").str.strip().str.lower() == "оплачен"].copy()
paid_clean["month"] = paid_clean["order_date"].dt.month

sep_revenue = paid_clean.loc[paid_clean["month"] == 9, "amount"].sum()
oct_revenue = paid_clean.loc[paid_clean["month"] == 10, "amount"].sum()
overall_change = (oct_revenue - sep_revenue) / sep_revenue * 100 if sep_revenue else np.nan

print("Все города:")
print("Сентябрь:", round(sep_revenue, 2))
print("Октябрь:", round(oct_revenue, 2))
print("Изменение:", round(overall_change, 2), "%")

city_changes = {}
for city in sorted(clean_df["city"].unique()):
    city_paid = paid_clean[paid_clean["city"] == city]
    sep = city_paid.loc[city_paid["month"] == 9, "amount"].sum()
    octo = city_paid.loc[city_paid["month"] == 10, "amount"].sum()
    change = (octo - sep) / sep * 100 if sep else np.nan
    city_changes[city] = change
    print(f"{city}: сентябрь={sep:.2f}, октябрь={octo:.2f}, изменение={change:.2f}%")

if city_changes:
    falling = {city: value for city, value in city_changes.items() if pd.notna(value)}
    if falling:
        worst_city = min(falling, key=falling.get)
        print("\\nСильнее всего падение в городе:", worst_city,
              "| изменение:", round(falling[worst_city], 2), "%")

print("\\nВывод для менеджера:")
print("Сравнение выполнено по очищенным данным: дубли заказов и TEST исключены, названия городов приведены к единому виду. Результат должен совпадать с таблицей урока 4, если там использовались те же правила очистки.")

Все города:
Сентябрь: 31783250.0
Октябрь: 30691160.0
Изменение: -3.44 %
Актобе: сентябрь=2368800.00, октябрь=2455540.00, изменение=3.66%
Алматы: сентябрь=13380590.00, октябрь=13517820.00, изменение=1.03%
Астана: сентябрь=8377290.00, октябрь=8087300.00, изменение=-3.46%
Караганда: сентябрь=3457520.00, октябрь=3045650.00, изменение=-11.91%
Шымкент: сентябрь=4199050.00, октябрь=3584850.00, изменение=-14.63%
\nСильнее всего падение в городе: Шымкент | изменение: -14.63 %
\nВывод для менеджера:
Сравнение выполнено по очищенным данным: дубли заказов и TEST исключены, названия городов приведены к единому виду. Результат должен совпадать с таблицей урока 4, если там использовались те же правила очистки.


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [4]:
# 3. Журнал очистки для менеджера
# Показываем число строк после каждого этапа и изменение общей выручки
# оплаченных заказов между сентябрём и октябрём.

def revenue_drop(data):
    paid_mask = data["status"].astype("string").str.strip().str.lower() == "оплачен"
    dates = pd.to_datetime(data["order_date"], errors="coerce")
    amounts = pd.to_numeric(data["amount"], errors="coerce")
    september = amounts[paid_mask & (dates.dt.month == 9)].sum()
    october = amounts[paid_mask & (dates.dt.month == 10)].sum()
    return (october - september) / september * 100 if september else np.nan

raw = pd.read_csv("orders.csv")
stages = [("Исходные данные", raw.copy())]

step = raw.copy()
step["amount"] = pd.to_numeric(step["amount"].astype("string").str.replace(r"\\s+", "", regex=True), errors="coerce")
step = step.dropna(subset=["amount"])
stages.append(("Преобразование amount и удаление пустых сумм", step.copy()))

step["city"] = step["city"].astype("string").str.strip().str.lower().map(CITY_MAP)
step = step.dropna(subset=["city"])
stages.append(("Нормализация city и удаление неизвестных городов", step.copy()))

step = step.drop_duplicates(subset=["order_id"], keep="first")
stages.append(("Удаление дублей order_id", step.copy()))

step["order_date"] = pd.to_datetime(step["order_date"], errors="coerce")
step = step.dropna(subset=["order_date"])
stages.append(("Преобразование дат и удаление некорректных дат", step.copy()))

step = step[step["customer_id"].astype("string").str.strip().str.upper() != "TEST"].copy()
stages.append(("Исключение TEST", step.copy()))

print("| Этап | Строк осталось | Удалено на этапе | Падение выручки, % |")
print("|---|---:|---:|---:|")
previous_count = len(raw)
for name, frame in stages:
    count = len(frame)
    removed = previous_count - count
    drop = revenue_drop(frame)
    print(f"| {name} | {count} | {removed} | {drop:.2f} |")
    previous_count = count

print("\\nВывод для менеджера:")
print("Наибольшее влияние на показатель видно по изменению процента падения между соседними этапами. Сравните строки таблицы: этап с самым большим изменением сильнее всего исказил исходную оценку.")

| Этап | Строк осталось | Удалено на этапе | Падение выручки, % |
|---|---:|---:|---:|
| Исходные данные | 5463 | 0 | -23.87 |
| Преобразование amount и удаление пустых сумм | 5018 | 445 | -23.87 |
| Нормализация city и удаление неизвестных городов | 4883 | 135 | -24.33 |
| Удаление дублей order_id | 4729 | 154 | -21.42 |
| Преобразование дат и удаление некорректных дат | 4729 | 0 | -21.42 |
| Исключение TEST | 4726 | 3 | -3.44 |
\nВывод для менеджера:
Наибольшее влияние на показатель видно по изменению процента падения между соседними этапами. Сравните строки таблицы: этап с самым большим изменением сильнее всего исказил исходную оценку.


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
# 4. Пропуски в сроках доставки
missing_total = clean_df["delivery_days"].isna().sum()
print("Пропусков delivery_days:", missing_total)

print("\\nПропуски по городам:")
print(clean_df.groupby("city")["delivery_days"].apply(lambda x: x.isna().sum()).sort_values(ascending=False))

print("\\nПропуски по статусам:")
print(clean_df.groupby("status")["delivery_days"].apply(lambda x: x.isna().sum()).sort_values(ascending=False))

# Анализ доставки в Шымкенте. Для доли берём только заказы, где срок доставки известен.
shymkent = clean_df[clean_df["city"] == "Шымкент"].copy()
delivery = pd.to_numeric(shymkent["delivery_days"], errors="coerce")
known = delivery.dropna()
share_dropna = (known <= 2).mean() * 100 if len(known) else np.nan

median_days = delivery.median()
filled_median = delivery.fillna(median_days)
share_median = (filled_median <= 2).mean() * 100 if len(filled_median) else np.nan

filled_zero = delivery.fillna(0)
share_zero = (filled_zero <= 2).mean() * 100 if len(filled_zero) else np.nan

print("\\nШымкент — доля доставки за 2 дня:")
print("(а) dropna:", round(share_dropna, 2), "%")
print("(б) fillna(медиана):", round(share_median, 2), "%")
print("(в) fillna(0):", round(share_zero, 2), "%")

print("\\nВывод для менеджера:")
print("Наиболее честный вариант для оценки фактической скорости — dropna: он использует только известные сроки и не выдумывает значения. Заполнение медианой искусственно добавляет типичный срок вместо неизвестного, а fillna(0) ошибочно считает заказы с пропуском доставленными за ноль дней.")

Пропусков delivery_days: 147
\nПропуски по городам:
city
Алматы       63
Астана       38
Караганда    23
Шымкент      20
Актобе        3
Name: delivery_days, dtype: int64
\nПропуски по статусам:
status
оплачен    135
отменён     12
Name: delivery_days, dtype: int64
\nШымкент — доля доставки за 2 дня:
(а) dropna: 30.55 %
(б) fillna(медиана): 29.69 %
(в) fillna(0): 32.49 %
\nВывод для менеджера:
Наиболее честный вариант для оценки фактической скорости — dropna: он использует только известные сроки и не выдумывает значения. Заполнение медианой искусственно добавляет типичный срок вместо неизвестного, а fillna(0) ошибочно считает заказы с пропуском доставленными за ноль дней.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [6]:
# 5. Исправление кода AI
df_ai = pd.read_csv("orders.csv")

# Ошибка 1: drop_duplicates() возвращает новую таблицу, поэтому результат нужно сохранить.
df_ai = df_ai.drop_duplicates(subset=["order_id"])

# Ошибка 2: lower() только меняет регистр, но не убирает пробелы
# и не объединяет русские/английские варианты написания. Используем словарь.
df_ai["city"] = (
    df_ai["city"].astype("string").str.strip().str.lower().map(CITY_MAP)
)

# Для сопоставимости с чистыми данными исключаем строки без распознанного города,
# некорректные суммы/даты и TEST.
df_ai["amount"] = pd.to_numeric(
    df_ai["amount"].astype("string").str.replace(r"\\s+", "", regex=True),
    errors="coerce"
)
df_ai["order_date"] = pd.to_datetime(df_ai["order_date"], errors="coerce")
df_ai = df_ai.dropna(subset=["city", "amount", "order_date"])
df_ai = df_ai[df_ai["customer_id"].astype("string").str.strip().str.upper() != "TEST"]

print("Количество городов:", df_ai["city"].nunique())
print("Количество строк:", len(df_ai))
print("Города:", sorted(df_ai["city"].unique()))

print("\\nОбъяснение:")
print("Первая ошибка — результат drop_duplicates() не был присвоен обратно df. Вторая — одного lower() недостаточно: нужно нормализовать пробелы и варианты названий городов по словарю.")

Количество городов: 5
Количество строк: 4726
Города: ['Актобе', 'Алматы', 'Астана', 'Караганда', 'Шымкент']
\nОбъяснение:
Первая ошибка — результат drop_duplicates() не был присвоен обратно df. Вторая — одного lower() недостаточно: нужно нормализовать пробелы и варианты названий городов по словарю.


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
# 6. Дни недели
weekday_names = clean_df["order_date"].dt.day_name()
weekday_counts = weekday_names.value_counts()

# Чтобы в выводе присутствовали все дни недели, включая дни с нулём заказов.
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]
weekday_counts = weekday_counts.reindex(weekday_order, fill_value=0)

print("Количество заказов по дням недели:")
print(weekday_counts)

least_day = weekday_counts.idxmin()
most_day = weekday_counts.idxmax()
least_count = weekday_counts.min()
most_count = weekday_counts.max()
difference_percent = (most_count - least_count) / most_count * 100 if most_count else np.nan

print("\\nМеньше всего заказов:", least_day, "-", least_count)
print("Больше всего заказов:", most_day, "-", most_count)
print("На", round(difference_percent, 2), "% меньше, чем в самый загруженный день.")

print("\\nВывод для менеджера:")
print("График курьеров стоит планировать с учётом недельной нагрузки: больше курьеров выводить в самый загруженный день, а в день с минимальным числом заказов можно уменьшить смену. При этом нужно учитывать не только количество заказов, но и часы пик.")

Количество заказов по дням недели:
order_date
Monday       617
Tuesday      719
Wednesday    680
Thursday     691
Friday       700
Saturday     699
Sunday       620
Name: count, dtype: int64
\nМеньше всего заказов: Monday - 617
Больше всего заказов: Tuesday - 719
На 14.19 % меньше, чем в самый загруженный день.
\nВывод для менеджера:
График курьеров стоит планировать с учётом недельной нагрузки: больше курьеров выводить в самый загруженный день, а в день с минимальным числом заказов можно уменьшить смену. При этом нужно учитывать не только количество заказов, но и часы пик.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
# 8